<a href="https://colab.research.google.com/github/DEMIGODXV5/Natural-Language-to-SQL-queries/blob/main/Colab_commit_Natural_Language_to_SQL_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hello world")

hello world


In [1]:
import numpy as np


In [2]:
! curl "https://api.mockaroo.com/api/dde01370?count=1000&key=11149690" > "customers.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 98952    0 98952    0     0  65551      0 --:--:--  0:00:01 --:--:-- 65574


In [3]:
! curl "https://api.mockaroo.com/api/8ba6f630?count=1000&key=11149690" > "products.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  257k    0  257k    0     0   153k      0 --:--:--  0:00:01 --:--:--  153k


In [4]:
! curl "https://api.mockaroo.com/api/6fa67fe0?count=3000&key=11149690" > "orders.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  229k    0  229k    0     0  37238      0 --:--:--  0:00:06 --:--:-- 57625


In [5]:
import sqlite3

In [26]:
import plotly.io as pio
pio.renderers.default = "colab" # or "png" if you want it to be a permanent image


In [8]:
import pandas as pd


In [9]:
import os

In [10]:
# Define SQL schemas for creating tables
customers_schema = """
CREATE TABLE IF NOT EXISTS customers (
customer_id INT PRIMARY KEY,
first_name VARCHAR(50),
last_name VARCHAR(50),
email VARCHAR(50),
phone_number VARCHAR(50),
address VARCHAR(50),
city VARCHAR(50),
country VARCHAR(50),
postal_code VARCHAR(50),
loyalty_points INT
);
"""

products_schema ="""
CREATE TABLE IF NOT EXISTS products (
product_id INT PRIMARY KEY,
product_name TEXT,
description TEXT,
price DECIMAL(10,2),
discount_percentage DECIMAL(5,2),
category VARCHAR(50),
brand TEXT,
stock_quantity INT,
color VARCHAR(50),
size VARCHAR(20),
weight DECIMAL(5,2),
dimensions TEXT,
release_date DATE,
rating DECIMAL(3,1),
reviews_count INT,
seller_name TEXT,
seller_rating DECIMAL(3,1),
seller_reviews_count INT,
shipping_method VARCHAR(20),
shipping_cost DECIMAL(6,2)

);
"""



orders_schema = """
CREATE TABLE IF NOT EXISTS orders (
order_id INT PRIMARY KEY,
customer_id INT,
product_id INT,
quantity INT,
unit_price DECIMAL(10,2),
total_price DECIMAL(10,2),
order_date DATE,
shipping_address VARCHAR(255),
payment_method VARCHAR(20),
status VARCHAR(20),
FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""

In [11]:
db_name='ecommerce.db'
if(os.path.exists(db_name)):
  os.remove(db_name)
  print(f"removed existing database '{db_name}'.")

In [12]:
import sqlite3
import pandas as pd
import os

COLUMN_DATA_TYPES={
    'customers': {
        'customer_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'email': 'object',
        'phone_number': 'object',
        'address': 'object',
        'city': 'object',
        'country': 'object',
        'postal_code': 'object',
        'loyalty_points': 'int64'

    },
    'products': {
        'product_id': 'int64',
        'product_name': 'object',
        'description': 'object',
        'price': 'float64',
        'discount_percentage': 'float64',
        'category': 'object',
        'brand': 'object',
        'stock_quantity': 'int64',
        'color': 'object',
        'size': 'object',
        'weight': 'float64',
        'dimensions': 'object',
        'release_date': 'datetime64 [ns] ',
        'rating': 'float64',
        'reviews_count': 'int64',
        'seller_name': 'object',
        'seller_rating': 'float64',
        'seller_reviews_count': 'int64',
        'shipping_method': 'object',
        'shipping_cost': 'float64'
    },

    'orders': {
        'order_id': 'int64',
        'customer_id': 'int64',
        'product_id': 'int64',
        'quantity': 'int64',
        'unit_price': 'float64',
        'total_price': 'float64',
        'order_date': 'datetime64[ns]',
        'shipping_address': 'object',
        'payment_method': 'object',
        'status': 'object'
    }

}

In [13]:
db_name='ecommerce.db'
conn=None # Initilaize the connection to None

try:
  conn=sqlite3.connect(db_name)
  cursor=conn.cursor()
  print(f"Database '{db_name}' created and connected successfully.")

  # create tables
  cursor.execute(customers_schema)
  cursor.execute(products_schema)
  cursor.execute(orders_schema)
  print("Tables 'customers','products', and 'orders' created successfully.")

  # load data from csv files into the tables using pandas
  csv_to_table_map={
      '/content/customers.csv': 'customers',
      '/content/products.csv': 'products',
      '/content/orders.csv': 'orders'
  }

  for csv_file , table_name in csv_to_table_map.items():
    if os.path.exists(csv_file):
      print(f"\n Processing {csv_file} for table {table_name} ...")

      # Read the CSV file into a pandas DataFrame
      df = pd.read_csv(csv_file)

      # 1. Get the expected schema for the current table
      expected_schema = COLUMN_DATA_TYPES[table_name]
      expected_cols = list(expected_schema.keys())

      # 2. Handle missing/extra columns
      # Drop columns from DataFrame that are not in the schema
      df=df[df.columns.intersection(expected_cols)]

      # Add any missing columns and fill with None (which becomes NULL in SQL)
      for col in expected_cols:
        if col not in df.columns:
          df [col] = None

      # 3. Reorder columns to match the defined schema exactly
      df = df [expected_cols]

      # 4. Enforce data types
      for col, dtype in expected_schema.items():
        if 'datetime' in dtype:
          # Use pd.to_datetime for date/time columns, coercing errors to NaT (Not a Time)
          df[col] = pd.to_datetime(df[col], errors='coerce')
        else:
          # Use astype for other columns, handling potential conversion errors
          try:
            df[col] = df[col].astype(dtype)
          except (ValueError, TypeError) as e:
            print(f" - Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")

      # Use the to_sql method to insert the cleaned DataFrame
      df.to_sql(table_name, conn, if_exists='append', index=False)
      print(f" -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
    else:
      print(f" Warning : {csv_file} not found. Skipping data load for {table_name} . ")

  #Commit the changes to the database
  conn.commit()
  print("\n data committed to the database successfully.")

except sqlite3.Error as e:
  print(f" Database Error: {e}")
except pd.errors.EmptyDataError as e:
  print(f"Pandas error:{e}. One of the CSV files moght be empty.")
except KeyError as e:
  print(f"Schema definition error: A column is missing from the TABLE_DATA_TYPES dictionary: {e}.")
except Exception as e:
  print(f"An unexpected error occurred: {e}")
finally:
  if conn:
    conn.close()
    print("Database connection closed.")



Database 'ecommerce.db' created and connected successfully.
Tables 'customers','products', and 'orders' created successfully.

 Processing /content/customers.csv for table customers ...
 -> Data from '/content/customers.csv' loaded into 'customers' table successfully.

 Processing /content/products.csv for table products ...
 -> Data from '/content/products.csv' loaded into 'products' table successfully.

 Processing /content/orders.csv for table orders ...
 -> Data from '/content/orders.csv' loaded into 'orders' table successfully.

 data committed to the database successfully.
Database connection closed.


In [15]:
#---- DATA WRANGLING REPORT ---
import pandas as pd

def show_data_health():
    tables = ['customers', 'products', 'orders']
    conn = sqlite3.connect('ecommerce.db')
    print("📊 DATASET HEALTH REPORT")
    for t in tables:
        df = pd.read_sql(f"SELECT * FROM {t}", conn)
        print(f"\nTable: {t.upper()}")
        print(f"- Total Records: {len(df)}")
        print(f"- Missing Values: {df.isnull().sum().sum()}")
        print(f"- Duplicates: {df.duplicated().sum()}")
    conn.close()

show_data_health()


📊 DATASET HEALTH REPORT

Table: CUSTOMERS
- Total Records: 1000
- Missing Values: 545
- Duplicates: 0

Table: PRODUCTS
- Total Records: 1000
- Missing Values: 1000
- Duplicates: 0

Table: ORDERS
- Total Records: 3000
- Missing Values: 0
- Duplicates: 0


In [ ]:
# !pip install google-genai

In [16]:
from google import genai

In [17]:
from google.colab import userdata

In [18]:
genai_client=genai.Client(api_key=userdata.get("new_key_gemini").strip())

In [24]:
prompt="""
### ROLE

You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation.

Your job is to convert natural language English queries into accurate, optimized SQLite queries for an e-commerce business intelligence dashboard used by non-technical users.

You must carefully understand user intent and generate SQL queries strictly based on the provided schema.

------------------------------------------------------------

### DATABASE SCHEMA

The database contains the following tables:

------------------------------------------------------------

TABLE: customers

Columns:
- customer_id (INTEGER PRIMARY KEY)
- first_name (TEXT)
- last_name (TEXT)
- email (TEXT)
- phone_number (TEXT)
- address (TEXT)
- city (TEXT)
- country (TEXT)
- postal_code (TEXT)
- loyalty_points (INTEGER)

------------------------------------------------------------

TABLE: products

Columns:
- product_id (INTEGER PRIMARY KEY)
- product_name (TEXT)
- description (TEXT)
- price (REAL)
- discount_percentage (REAL)
- category (TEXT)
- brand (TEXT)
- stock_quantity (INTEGER)
- color (TEXT)
- size (TEXT)
- weight (REAL)
- dimensions (TEXT)
- release_date (DATE)
- rating (REAL)
- reviews_count (INTEGER)
- seller_name (TEXT)
- seller_rating (REAL)
- seller_reviews_count (INTEGER)
- shipping_method (TEXT)
- shipping_cost (REAL)

------------------------------------------------------------

TABLE: orders

Columns:
- order_id (INTEGER PRIMARY KEY)
- customer_id (INTEGER)
- product_id (INTEGER)
- quantity (INTEGER)
- unit_price (REAL)
- total_price (REAL)
- order_date (DATE)
- shipping_address (TEXT)
- payment_method (TEXT)
- status (TEXT)

------------------------------------------------------------

### TABLE RELATIONSHIPS

customers.customer_id → orders.customer_id

products.product_id → orders.product_id

------------------------------------------------------------

### PRIMARY TASK

Convert the user's natural language query into a valid SQLite query.

Understand what the user is asking.

Identify relevant tables.

Identify required joins.

Generate optimized SQL query.

------------------------------------------------------------

### QUERY RULES

Generate ONLY SELECT queries.

NEVER generate:
- INSERT
- UPDATE
- DELETE
- DROP
- ALTER
- TRUNCATE

Use only tables listed above.

Use only columns listed above.

Never invent:
- tables
- columns
- relationships

Use proper JOIN statements when multiple tables are needed.

------------------------------------------------------------

### BUSINESS LOGIC RULES

Revenue → use total_price

Product sales → use quantity

Customer purchase analysis → join customers + orders

Product performance → join products + orders

Seller analysis → use seller_name, seller_rating

Discount analysis → use discount_percentage

Shipping analysis → use shipping_method, shipping_cost

Customer loyalty → use loyalty_points

Product ratings → use rating and reviews_count

------------------------------------------------------------

### AGGREGATION RULES

Use:
- SUM()
- COUNT()
- AVG()
- MAX()
- MIN()

Use GROUP BY when aggregation is required.

Use ORDER BY when ranking/sorting is needed.

Use LIMIT for top/bottom requests.

------------------------------------------------------------

### DATE HANDLING RULES

If user asks "today":

DATE(order_date) = DATE('now')

------------------------------------------------------------

If user asks "this month":

strftime('%Y-%m', order_date) = strftime('%Y-%m', 'now')

------------------------------------------------------------

If user asks "last month":

strftime('%Y-%m', order_date) = strftime('%Y-%m', date('now', '-1 month'))

------------------------------------------------------------

If user asks "this year":

strftime('%Y', order_date) = strftime('%Y', 'now')

------------------------------------------------------------

### AMBIGUITY HANDLING

If the user query is unclear, incomplete, or ambiguous:

Return clarification request.

Examples of ambiguous requests:
- Show best customers
- Show top products
- Show recent orders
- Show high-performing sellers

Ask follow-up question when needed.

------------------------------------------------------------

### ERROR HANDLING

Return an error response if:

- User asks for unavailable tables
- User asks for unavailable columns
- User asks for unsupported metrics
- Query cannot be generated

------------------------------------------------------------

### OUTPUT FORMAT

Your final response must be a single valid JSON object with the following keys:

1. "status"

A string with one of these values:
- "success"
- "clarification_needed"
- "error"

------------------------------------------------------------

2. "response"

If status is "success":
Return complete SQLite query string.

If status is "clarification_needed":
Return follow-up clarification question.

If status is "error":
Return reason query could not be generated.

------------------------------------------------------------

### RESPONSE RULES

- Return valid JSON only
- Return only one JSON object
- No markdown
- No code blocks
- No explanations
- No comments
- No extra keys

------------------------------------------------------------

### EXAMPLES

Example 1

User:
Show top 5 customers by total spending

Response:
{
  "status": "success",
  "response": "SELECT c.first_name, c.last_name, SUM(o.total_price) AS total_spent FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id ORDER BY total_spent DESC LIMIT 5;"
}

------------------------------------------------------------

Example 2

User:
Show best customers

Response:
{
  "status": "clarification_needed",
  "response": "Do you want best customers based on total spending, total orders, or loyalty points?"
}

------------------------------------------------------------

Example 3

User:
Show employee salaries

Response:
{
  "status": "error",
  "response": "The requested data does not exist in the current database schema."
}

------------------------------------------------------------

IMPORTANT FINAL INSTRUCTION

Generate accurate SQLite queries strictly based on the provided schema.

Never hallucinate columns/tables.

Always return valid JSON output.
"""

In [19]:
import json
def get_sql_query(genai_client, prompt, user_query) :

  # https://www.geeksforgeeks.org/python/formatted-string-literals-f-strings-python/
  contents = f"""
  {prompt}
  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash-lite', contents=contents)

  # Access the usage_metadata attribute
  usage_metadata = response. usage_metadata

  # Print the different token counts
  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata. candidates_token_count}")
  print(f"Total Token Count: {usage_metadata. total_token_count}")

  output=json.loads(response.text.replace('```json', '').replace('```', ''))
  return output

In [20]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='ecommerce.db'):

  conn = None
  try:
    # Connect to the database
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor ()

    # Execute the query
    print(f"\nExecuting query on '{db_name}':\n{query}")
    cursor.execute (query)

    # Fetch all results
    results = cursor. fetchall()

    # Get column names from the cursor description
    columns = [description [0] for description in cursor.description]

    # Format results as a dataframe for easier use
    results_as_dict = [dict(zip(columns, row)) for row in results]
    results_df = pd.DataFrame(results_as_dict)

    print("Query executed successfully.")
        # ---  DESCRIPTIVE STATISTICS ---
    if not results_df.empty:
        print("\n📈 ANALYTICS SUMMARY:")
        numeric_cols = results_df.select_dtypes(include=['number']).columns
        if not numeric_cols.empty:
            # Shows Mean, Max, Min for any numbers found in the result
            display(results_df[numeric_cols].describe().loc[['mean', 'max', 'min']])


    return results_df

  except sqlite3.Error as e:
    print(f"Database error executing query: {e}")
    return None
  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    return None
  finally:
    if conn:
      conn.close()

In [29]:
# def text2sql(genai_client, prompt, user_query):
#   output=get_sql_query(genai_client,prompt,user_query)
#   if output['status']=='success':
#     results=execute_query(output['response'])
#     return results
#   return output
# def text2sql(genai_client, prompt, user_query):
#     output = get_sql_query(genai_client, prompt, user_query)
#     if output['status'] == 'success':
#         results = execute_query(output['response'])
#         # AUTO-VISUALIZE CALL
#         visualize_results(results, user_query)
#         return results
#     return output

def text2sql(genai_client, prompt, user_query):
    output = get_sql_query(genai_client, prompt, user_query)
    if output['status'] == 'success':
        results = execute_query(output['response'])

        if results is not None and not results.empty:
            # CALL THE OUTLIER FUNCTION HERE
            detect_outliers(results)

            visualize_results(results, user_query)

        return results
    return output





In [22]:
import plotly.express as px

def visualize_results(df, user_query):
    if isinstance(df, pd.DataFrame) and not df.empty:
        # DSBDA FIX: Aggregate data to avoid duplicate bars/weird scaling
        cols = df.columns
        if len(cols) >= 2:
            # Group by the first column and sum the second
            agg_df = df.groupby(cols[0])[cols[1]].sum().reset_index()

            # Sort it so the highest bar is first (looks better for DSBDA)
            agg_df = agg_df.sort_values(by=cols[1], ascending=False).head(15)

            fig = px.bar(agg_df, x=cols[0], y=cols[1],
                         title=f"Top Results for: {user_query}")
            fig.show()





In [28]:
def detect_outliers(df):
    if df is None or df.empty:
        return

    numeric_df = df.select_dtypes(include=['number'])
    if numeric_df.empty:
        return

    print("\n🚨 DSBDA OUTLIER REPORT (IQR Method):")
    for col in numeric_df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filtering the rows that are outliers
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

        if not outliers.empty:
            print(f"- Column '{col}': Found {len(outliers)} outliers (Values outside {lower_bound:.2f} to {upper_bound:.2f})")
            # This displays the table you see in your photo
            display(outliers)
        else:
            print(f"- Column '{col}': No outliers detected.")


In [30]:
# Run this in a new cell
query = "What is the average price of products in each category?"
res = text2sql(genai_client, prompt, query)


Input Token Count: 1431
Thoughts Token Count: None
Output Token Count: 34
Total Token Count: 1465

Executing query on 'ecommerce.db':
SELECT category, AVG(price) AS average_price FROM products GROUP BY category;
Query executed successfully.

📈 ANALYTICS SUMMARY:


,average_price
mean,488.821779
max,503.244091
min,457.134688



🚨 DSBDA OUTLIER REPORT (IQR Method):
- Column 'average_price': Found 1 outliers (Values outside 467.36 to 518.06)


,category,average_price
4,toys,457.134688


In [31]:
query = "show me the order count by country ascending to descending"
res = text2sql(genai_client, prompt, query)

Input Token Count: 1430
Thoughts Token Count: None
Output Token Count: 64
Total Token Count: 1494

Executing query on 'ecommerce.db':
SELECT c.country, COUNT(o.order_id) AS order_count FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.country ORDER BY order_count DESC;
Query executed successfully.

📈 ANALYTICS SUMMARY:


,order_count
mean,25.423729
max,576.000000
min,1.000000



🚨 DSBDA OUTLIER REPORT (IQR Method):
- Column 'order_count': Found 16 outliers (Values outside -19.50 to 40.50)


,country,order_count
0,China,576
1,Indonesia,311
2,Philippines,172
3,Russia,148
4,France,145
5,Poland,99
6,Portugal,98
7,Brazil,84
8,United States,78
9,Ukraine,76


In [32]:
text2sql(genai_client,prompt , "What are my most popular products based on total sales quantity")
# res = text2sql(genai_client, prompt, "show me order count by country")
# visualize_results(res, "order count by country")

Input Token Count: 1431
Thoughts Token Count: None
Output Token Count: 70
Total Token Count: 1501

Executing query on 'ecommerce.db':
SELECT p.product_name, SUM(o.quantity) AS total_quantity_sold FROM products p JOIN orders o ON p.product_id = o.product_id GROUP BY p.product_id ORDER BY total_quantity_sold DESC;
Query executed successfully.

📈 ANALYTICS SUMMARY:


,total_quantity_sold
mean,155.93299
max,541.00000
min,2.00000



🚨 DSBDA OUTLIER REPORT (IQR Method):
- Column 'total_quantity_sold': Found 12 outliers (Values outside -84.12 to 382.88)


,product_name,total_quantity_sold
0,eu mi nulla ac,541
1,praesent id massa id,459
2,nullam orci,432
3,non velit donec,424
4,mauris sit amet eros,423
5,elementum pellentesque quisque porta,420
6,sem sed sagittis nam,408
7,ac consequat,402
8,magna ac,398
9,amet sem fusce consequat,390


,product_name,total_quantity_sold
0,eu mi nulla ac,541
1,praesent id massa id,459
2,nullam orci,432
3,non velit donec,424
4,mauris sit amet eros,423
...,...,...
965,lorem integer tincidunt ante,4
966,lorem ipsum,4
967,donec posuere metus vitae,3
968,dui vel sem,3


In [33]:
# text2sql(genai_client,prompt , "which country do i sell the most to")
query = "which country do i sell the most to"
res = text2sql(genai_client, prompt, query)

Input Token Count: 1428
Thoughts Token Count: None
Output Token Count: 64
Total Token Count: 1492

Executing query on 'ecommerce.db':
SELECT c.country, COUNT(o.order_id) AS total_orders FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.country ORDER BY total_orders DESC;
Query executed successfully.

📈 ANALYTICS SUMMARY:


,total_orders
mean,25.423729
max,576.000000
min,1.000000



🚨 DSBDA OUTLIER REPORT (IQR Method):
- Column 'total_orders': Found 16 outliers (Values outside -19.50 to 40.50)


,country,total_orders
0,China,576
1,Indonesia,311
2,Philippines,172
3,Russia,148
4,France,145
5,Poland,99
6,Portugal,98
7,Brazil,84
8,United States,78
9,Ukraine,76


In [34]:
text2sql(genai_client,prompt , "Product with highest sales last month")

Input Token Count: 1426
Thoughts Token Count: None
Output Token Count: 39
Total Token Count: 1465


{'status': 'clarification_needed',
 'response': 'Do you want the product with the highest total quantity sold or highest total revenue generated last month?'}

In [35]:
text2sql(genai_client,prompt , "populate the product name to temp col name and if col name deosnt exist create a temp_product col")

Input Token Count: 1442
Thoughts Token Count: None
Output Token Count: 39
Total Token Count: 1481


{'status': 'error',
 'response': 'The query is unclear. Please specify what information you would like to retrieve related to product names and temporary columns.'}

In [38]:
text2sql(genai_client,prompt , "avg no of orders per day per country for january month ")

Input Token Count: 1432
Thoughts Token Count: None
Output Token Count: 102
Total Token Count: 1534

Executing query on 'ecommerce.db':
SELECT c.country, CAST(COUNT(o.order_id) AS REAL) / COUNT(DISTINCT DATE(o.order_date)) FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE strftime('%Y-%m', o.order_date) = strftime('%Y-%m', 'now', 'start of month') GROUP BY c.country;
Query executed successfully.


""


In [ ]:
query = "avg no of orders per day per country"
res = text2sql(genai_client, prompt, query)

Input Token Count: 1428
Thoughts Token Count: None
Output Token Count: 72
Total Token Count: 1500

Executing query on 'ecommerce.db':
SELECT c.country, CAST(COUNT(o.order_id) AS REAL) / COUNT(DISTINCT DATE(o.order_date)) FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.country;
Query executed successfully.

📈 ANALYTICS SUMMARY:


,CAST(COUNT(o.order_id) AS REAL) / COUNT(DISTINCT DATE(o.order_date))
mean,1.012173
max,1.304933
min,1.000000



🚨 DSBDA OUTLIER REPORT (IQR Method):
- Column 'CAST(COUNT(o.order_id) AS REAL) / COUNT(DISTINCT DATE(o.order_date))': Found 23 outliers (Values outside 1.00 to 1.00)


,country,CAST(COUNT(o.order_id) AS REAL) / COUNT(DISTINCT DATE(o.order_date))
3,Argentina,1.025000
13,Brazil,1.026316
16,Canada,1.021277


In [ ]:
text2sql(genai_client,prompt , "which country has least sales")

Input Token Count: 1425
Thoughts Token Count: 182
Output Token Count: 61
Total Token Count: 1668

Executing query on 'ecommerce.db':
SELECT c.country, SUM(o.total_price) AS total_sales FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.country ORDER BY total_sales ASC LIMIT 1;
Query executed successfully.


,country,total_sales
0,Kenya,309.18


In [ ]:
text2sql(genai_client,prompt , "Product with hig est salert last month")

Input Token Count: 1428
Thoughts Token Count: 422
Output Token Count: 91
Total Token Count: 1941

Executing query on 'ecommerce.db':
SELECT p.product_name FROM products p JOIN orders o ON p.product_id = o.product_id WHERE strftime('%Y-%m', o.order_date) = strftime('%Y-%m', date('now', '-1 month')) GROUP BY p.product_name ORDER BY SUM(o.quantity) DESC LIMIT 1;
Query executed successfully.


""


In [ ]:
text2sql(genai_client,prompt , "which count ry has lest sales")

Input Token Count: 1426
Thoughts Token Count: 171
Output Token Count: 67
Total Token Count: 1664

Executing query on 'ecommerce.db':
SELECT c.country, SUM(o.total_price) AS total_sales FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.country ORDER BY total_sales ASC LIMIT 1;
Query executed successfully.


,country,total_sales
0,Kenya,309.18


In [ ]:
text2sql(genai_client,prompt , "rank the products")

Input Token Count: 1423
Thoughts Token Count: 268
Output Token Count: 39
Total Token Count: 1730


{'status': 'clarification_needed',
 'response': 'Do you want to rank products by total quantity sold, total revenue generated, or average rating?'}

In [ ]:
text2sql(genai_client,prompt , "which country ranks in middle by total sales")

In [ ]:
text2sql(genai_client,prompt , "which country ranks in middle by total sales figure out somme way to find that middle country")

In [ ]:
text2sql(genai_client,prompt , "which  rank is India  in  by total sales")